In [ ]:
#####################################################
#
# REGRESIÓN LOGÍSTICA sobre datos ya preprocesados (PCA + encodes)
# - Entrena pipeline: drop de base para dummies + escalado + logística
# - Calcula α óptimo (criterio F1) en TEST y guarda política de inferencia
# - Exporta scores de train/test y empaqueta en un ZIP sin fechas
#
# Requiere:
#   - T_train_final_objetivo.csv
#   - T_test_final_objetivo.csv
#
# Devuelve (en carpeta mi_regresion_logistica/):
#   - modelo_reg_logistica.pkl
#   - expected_columns.json
#   - inference_policy.json
#   - T_train_final_objetivo_scores.csv
#   - T_test_final_objetivo_scores.csv
#   - mi_reg_logistica_artifacts_bundle.zip
#####################################################

import os, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

plt.rcParams["figure.dpi"] = 110

# ===== Lectura =====
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

# Separar X | y (última columna = objetivo binario)
X_train = Train.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].astype(int).to_numpy()
X_test  = Test.iloc[:, :-1].copy()
y_test  = Test.iloc[:, -1].astype(int).to_numpy()

# ===== Utilidades para dummies por prefijo "___" =====
SEP = "___"

def is_binary_series(s: pd.Series):
    vals = pd.unique(s.dropna())
    return set(vals).issubset({0,1}) or set(vals).issubset({0.0,1.0})

def prefix_of(col: str, sep=SEP):
    return col.split(sep, 1)[0] if sep in col else None

def build_nominal_blocks_by_prefix(X: pd.DataFrame, sep=SEP):
    blocks = {}
    for c in X.columns:
        if sep in c and is_binary_series(X[c]):
            blocks.setdefault(prefix_of(c, sep), []).append(c)
    # respetar orden del CSV
    for k, v in blocks.items():
        blocks[k] = [c for c in X.columns if c in set(v)]
    return blocks

# ===== Precomputo de bases a dropear (una por bloque) =====
blocks = build_nominal_blocks_by_prefix(X_train, SEP)
drop_cols = [cols[0] for cols in blocks.values() if len(cols) >= 2]  # primera de cada bloque

# ===== ColumnTransformer: ARREGLAR_DESPEJE (con fallback de compatibilidad) =====
try:
    arreglar_despeje = ColumnTransformer(
        transformers=[("drop_nominal_bases", "drop", drop_cols)],
        remainder="passthrough",
        verbose_feature_names_out=False,
        force_int_remainder_cols=False  # <- si tu sklearn lo admite, evita warning en tu entorno
    )
except TypeError:
    # fallback estándar (sklearn oficial no tiene force_int_remainder_cols)
    arreglar_despeje = ColumnTransformer(
        transformers=[("drop_nominal_bases", "drop", drop_cols)],
        remainder="passthrough",
        verbose_feature_names_out=False
    )

# ===== Pipeline completo =====
mi_regresion_logistica = Pipeline(steps=[
    ("dropper", arreglar_despeje),                 # <- se mantiene el nombre/etapa
    ("scaler", StandardScaler(with_mean=False)),
    ("logit", LogisticRegression(
        solver="liblinear",
        penalty="l2",
        max_iter=2000,
        class_weight="balanced",
        fit_intercept=True,
        random_state=0
    ))
])

# ===== Entrenamiento =====
mi_regresion_logistica.fit(X_train, y_train)

# ===== Scores (prob clase positiva) =====
p_train = mi_regresion_logistica.predict_proba(X_train)[:, 1]
p_test  = mi_regresion_logistica.predict_proba(X_test)[:, 1]
Train_out = Train.copy(); Train_out["scores"] = p_train
Test_out  = Test.copy();  Test_out["scores"]  = p_test
Train_out.to_csv("T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv("T_test_final_objetivo_scores.csv", index=False)

# ===== Barrido de umbrales (TEST) =====
def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0, 1.0, 0.01)):
    rows = []
    for alpha in thresholds:
        y_pred = (probs >= alpha).astype(int)
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary', zero_division=0
        )
        rows.append({"threshold": alpha, "accuracy": acc, "precision": prec, "recall": rec, "f1_score": f1})
    return pd.DataFrame(rows)

threshold_results = evaluate_thresholds(y_test, p_test)

print("=== MEJORES UMBRALES EN TEST ===")
print(f"Mejor F1-score:  {threshold_results.loc[threshold_results['f1_score'].idxmax(), 'threshold']:.3f}")
print(f"Mejor Accuracy:  {threshold_results.loc[threshold_results['accuracy'].idxmax(), 'threshold']:.3f}")
print(f"Mejor Precision: {threshold_results.loc[threshold_results['precision'].idxmax(), 'threshold']:.3f}")
print(f"Mejor Recall:    {threshold_results.loc[threshold_results['recall'].idxmax(), 'threshold']:.3f}")

# ===== Selección de α óptimo (criterio F1) =====
from sklearn.metrics import roc_curve, precision_recall_curve

def find_optimal_threshold(y_true, probs, method='f1'):
    if method == 'f1':
        precision, recall, thresholds = precision_recall_curve(y_true, probs)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        idx = np.argmax(f1_scores[:-1]) if len(f1_scores) > 1 else 0
        return float(thresholds[idx]) if len(thresholds) > 0 else 0.5
    elif method == 'youden':
        fpr, tpr, thresholds = roc_curve(y_true, probs)
        return float(thresholds[np.argmax(tpr - fpr)])
    elif method == 'accuracy':
        grid = np.arange(0.0, 1.0, 0.01)
        accs = [(t, accuracy_score(y_true, (probs >= t).astype(int))) for t in grid]
        return float(max(accs, key=lambda x: x[1])[0])
    else:
        return 0.5

alpha_optimo = find_optimal_threshold(y_test, p_test, method='f1')
print(f"\n=== EVALUACIÓN FINAL CON α* (F1) = {alpha_optimo:.3f} ===")
y_pred_optimo = (p_test >= alpha_optimo).astype(int)

acc_final = accuracy_score(y_test, y_pred_optimo)
prec_final, rec_final, f1_final, _ = precision_recall_fscore_support(
    y_test, y_pred_optimo, average='binary', zero_division=0
)
cm = confusion_matrix(y_test, y_pred_optimo)

print(f"Accuracy: {acc_final:.3f}")
print(f"Precision: {prec_final:.3f}")
print(f"Recall: {rec_final:.3f}")
print(f"F1-score: {f1_final:.3f}")
print("Matriz de confusión:\n", cm)

# ===== Guardado del modelo y artefactos =====
import joblib
joblib.dump(mi_regresion_logistica, "modelo_reg_logistica.pkl")  # pipeline completo

with open("expected_columns.json", "w", encoding="utf-8") as f:
    json.dump({"columns": X_train.columns.tolist(),
               "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f, ensure_ascii=False, indent=2)

# ===== Política de inferencia (usa α óptimo y clase positiva) =====
classes_train = sorted(pd.unique(y_train))
pos_label = 1 if 1 in classes_train else max(classes_train)
inference_policy = {
    "task": "binary",
    "decision": {
        "type": "threshold",
        "alpha": float(alpha_optimo),
        "criterion": "f1",
        "pos_label": str(pos_label)
    },
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}
with open("inference_policy.json", "w", encoding="utf-8") as f:
    json.dump(inference_policy, f, ensure_ascii=False, indent=2)

print("Artefactos guardados:",
      "modelo_reg_logistica.pkl",
      "expected_columns.json",
      "inference_policy.json",
      "T_train_final_objetivo_scores.csv",
      "T_test_final_objetivo_scores.csv")

# ===== Bundle ZIP =====
dst_dir = Path("mi_regresion_logistica"); dst_dir.mkdir(exist_ok=True)
zip_path = dst_dir / "mi_reg_logistica_artifacts_bundle.zip"

candidates = [
    "modelo_reg_logistica.pkl",
    "expected_columns.json",
    "inference_policy.json",
    "T_train_final_objetivo_scores.csv",
    "T_test_final_objetivo_scores.csv",
]
present = [f for f in candidates if Path(f).exists()]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=Path(f).name)

print("\nZIP creado en:", zip_path.resolve())
print("Incluidos:", [Path(f).name for f in present])
